In [0]:
# IMPORTS
import logging
from pyspark.sql import DataFrame, SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException
from databricks.sdk.runtime import *

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")
import aws_utils
# Get Environment Prefix
env = aws_utils.get_env_prefix()

In [0]:
#Take the data of the latest external file!!!

from pyspark.sql import functions as F

source_table = f"`{env}-cdh`.externalfiles.exthoist_indexl3869"
target_table = f"`{env}-reportinputfiles`.investor.hoist_indexl3869"

# 1. Βρίσκουμε το max ingestion timestamp
max_ts = spark.sql(f"""
SELECT MAX(IngestionTimestamp) as max_ts
FROM {source_table}
""").collect()[0]["max_ts"]

# 2. Παίρνουμε το pool των τελευταίων records
df_latest = spark.sql(f"""
SELECT PortfolioName,
       TaxNumber,
       LeverageHoist,
       FinalDecision,
       ExpectedHaircut,
       1 as IsLatestFlag,
       cast(BusinessDate as timestamp) ReferenceDate,
       IngestionTimestamp
FROM {source_table}
WHERE IngestionTimestamp = '{max_ts}'
""")

df_latest.createOrReplaceTempView("src_latest")

In [0]:
# Βρίσκουμε υπάρχοντα Latest records του `{env}-reportinputfiles`.investor.hoist_indexl3869
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tgt_latest AS
SELECT *
FROM {target_table}
WHERE IsLatestFlag = 1
""")

In [0]:
#Update τα παλιά latest σε 0 με βάση το κλειδί PortfolioName + TaxNumber
spark.sql(f"""
MERGE INTO {target_table} tgt
USING src_latest src
ON tgt.PortfolioName = src.PortfolioName
AND tgt.TaxNumber = src.TaxNumber
AND tgt.IsLatestFlag = 1

WHEN MATCHED THEN
UPDATE SET tgt.IsLatestFlag = 0
""")

In [0]:
df_latest.write \
    .mode("append") \
    .insertInto(target_table)